# California Housing : une régression sur des données réelles

Ce notebook accompagne l'exercice 7.3. Nous comparons une machine affine, une régression régularisée, une SVR et un MLP sur huit caractéristiques. Ici, aucune fonction exacte $f_*$ n'est disponible : toutes les erreurs calculées restent empiriques.

## Parcours

1. [Chargement et partition](#donnees-california)
2. [Machines affines et SVR](#classiques-california)
3. [MLP avec Flax NNX](#mlp-california)
4. [Comparaison et résidus](#residus-california)
5. [Tube et vecteurs supports](#epsilon-california)

In [1]:
from time import perf_counter

import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

JAX 0.11.1, Flax 0.12.9, Optax 0.2.8, scikit-learn 1.9.0


In [2]:
MODE_RAPIDE = True

if MODE_RAPIDE:
    N_DONNEES = 6_000
    EPOQUES_MLP = 45
else:
    N_DONNEES = None
    EPOQUES_MLP = 90

TAILLE_LOT = 128

<a id="donnees-california"></a>
## 1. Chargement et partition

Chargez le jeu `California Housing`. Le premier chargement nécessite une connexion internet ; les suivants utilisent le cache de `scikit-learn`. Le mode rapide sélectionne 6 000 observations avant de former les ensembles d'apprentissage, de validation et de test.

Normalisez chaque caractéristique avec la moyenne et l'écart-type calculés **uniquement sur l'apprentissage**.

In [ ]:
# À compléter.

<a id="classiques-california"></a>
## 2. Machines affines et SVR

Ajustez une machine affine, une machine affine régularisée et plusieurs SVR à noyau gaussien. Choisissez les hyperparamètres uniquement à partir de la perte de validation.

In [4]:
def indicateurs(z, prediction):
    residu = prediction - z
    return {
        "RMSE": float(np.sqrt(np.mean(residu**2))),
        "MAE": float(np.mean(np.abs(residu))),
    }


def choisir(candidats):
    return min(candidats, key=lambda r: (r[0], r[1]))

In [ ]:
# À compléter.

<a id="mlp-california"></a>
## 3. MLP avec Flax NNX

Écrivez un MLP de deux couches cachées. Pour stabiliser l'optimisation, normalisez aussi la cible avec les seules données d'apprentissage ; les prédictions sont ensuite remises dans l'unité d'origine.

In [6]:
class MLP(nnx.Module):
    def __init__(self, largeur, *, rngs):
        self.W1 = nnx.Linear(8, largeur, rngs=rngs)
        self.W2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.W3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        x = nnx.relu(self.W1(x))
        x = nnx.relu(self.W2(x))
        return self.W3(x).reshape(-1)


def perte_mlp(machine, x, z):
    return jnp.mean((machine(x) - z) ** 2)


@nnx.jit
def pas_mlp(machine, optimizer, x, z):
    valeur, gradient = nnx.value_and_grad(perte_mlp)(machine, x, z)
    optimizer.update(machine, gradient)
    return valeur


def entrainer_mlp(machine, x, z, *, alpha=1e-3, epoques=EPOQUES_MLP, graine=0):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    generateur = np.random.default_rng(graine)
    historique = []
    debut = perf_counter()
    for _ in range(epoques):
        permutation = generateur.permutation(z.size)
        pertes = []
        for i in range(0, z.size, TAILLE_LOT):
            indices = permutation[i:i + TAILLE_LOT]
            pertes.append(float(pas_mlp(
                machine, optimizer, jnp.asarray(x[indices]), jnp.asarray(z[indices])
            )))
        historique.append(float(np.mean(pertes)))
    return historique, perf_counter() - debut


def predire_mlp(machine, x):
    prediction_normalisee = np.asarray(machine(jnp.asarray(x)))
    return moyenne_z + ecart_z * prediction_normalisee

In [ ]:
# À compléter.

<a id="residus-california"></a>
## 4. Comparaison et résidus

Comparez RMSE et MAE sur les trois ensembles. Représentez les résidus de test en fonction de la prédiction. Une structure visible dans les résidus signale une erreur systématique de la machine — ou une particularité des données — que la seule moyenne cache.

In [ ]:
# À compléter.

<a id="epsilon-california"></a>
## 5. Tube et vecteurs supports

Avec $C$ et le noyau fixés, faites varier $\epsilon$. Comparez la fréquence des vecteurs supports, la RMSE et la MAE de validation et de test.

In [ ]:
# À compléter.

## Bilan

La normalisation et la séparation apprentissage-validation-test font partie de l'expérience mathématique. La SVR reste très compétitive pour ce jeu de taille modérée ; le MLP offre une classe de fonctions plus flexible, mais son optimisation introduit d'autres choix. Sans fonction exacte, l'analyse des résidus complète indispensablement les moyennes d'erreur.